In [ ]:
import pandas as pd

# Load the FINRA 2024 survey dataset
df = pd.read_csv("../data/NFCS 2024 State Data 250623.csv")

In [ ]:

# Display the dataset dimensions
print("Dataset shape:", df.shape)

# Check the columns needed for our analysis
columns = ["STATEQ", "B14A_1", "wgt_s3"]

# Check for missing values
print("\nMissing values:")
print(df[columns].isna().sum())


In [ ]:

# Count responses to the investment ownership question
print("Investment ownership responses:")
print(df["B14A_1"].value_counts(dropna=False).sort_index())

# Check the number of unique state codes
print("\nNumber of unique state codes:", df["STATEQ"].nunique())

# Examine the survey weight distribution
print("\nSurvey weight summary:")
print(df["wgt_s3"].describe())


In [ ]:

# Create a copy of the dataset for analysis
investments = df.copy()

# Convert investment responses to numeric values
investments["B14A_1"] = pd.to_numeric(
    investments["B14A_1"], errors="coerce"
)

# Keep only valid Yes (1) and No (2) responses
investments = investments[
    investments["B14A_1"].isin([1, 2])
].copy()

# Display the cleaned dataset size and response counts
print("Cleaned dataset shape:", investments.shape)
print("\nValid investment responses:")
print(investments["B14A_1"].value_counts().sort_index())

# Verify the number of states remaining
print("\nStates represented:", investments["STATEQ"].nunique())


In [ ]:

# Create a binary investment ownership indicator
investments["owns_investments"] = (
    investments["B14A_1"] == 1
).astype(int)

# Verify the new indicator
print("Investment ownership indicator:")
print(investments["owns_investments"].value_counts())

# Confirm all state survey weights are positive
print("\nAll state weights positive:",
      (investments["wgt_s3"] > 0).all())


In [ ]:

# Calculate the weighted ownership rate for each state
state_summary = (
    investments.groupby("STATEQ")
    .apply(
        lambda group: (
            (group["owns_investments"] * group["wgt_s3"]).sum()
            / group["wgt_s3"].sum()
        ) * 100,
        include_groups=False
    )
    .reset_index(name="participation_rate")
)

# Display the calculated state participation rates
print("Number of states:", len(state_summary))
print("\nState investment participation rates:")
print(state_summary.round(2).head(10))


In [ ]:

# Display all state codes in ascending order
print("State codes:")
print(sorted(state_summary["STATEQ"].unique()))

# Check the range and number of unique codes
print("\nMinimum state code:", state_summary["STATEQ"].min())
print("Maximum state code:", state_summary["STATEQ"].max())
print("Number of unique codes:", state_summary["STATEQ"].nunique())


In [ ]:

# State abbreviations in the official FINRA STATEQ order (1–51)
state_abbreviations = [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "DC",
    "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS", "KY",
    "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT",
    "NE", "NV", "NH", "NJ", "NM", "NY", "NC", "ND", "OH",
    "OK", "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT",
    "VT", "VA", "WA", "WV", "WI", "WY"
]

# Full state names in the same order (used for map tooltips)
state_names = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado",
    "Connecticut", "Delaware", "District of Columbia", "Florida", "Georgia",
    "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky",
    "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota",
    "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire",
    "New Jersey", "New Mexico", "New York", "North Carolina", "North Dakota",
    "Ohio", "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island",
    "South Carolina", "South Dakota", "Tennessee", "Texas", "Utah", "Vermont",
    "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming"
]
assert len(state_names) == len(state_abbreviations) == 51

# Map FINRA state codes to their two-letter abbreviations
state_mapping = dict(enumerate(state_abbreviations, start=1))

state_summary["state_code"] = state_summary["STATEQ"].map(state_mapping)
state_summary["state_name"] = state_summary["STATEQ"].map(
    dict(enumerate(state_names, start=1))
)

# Verify the mapping
print("Number of mapped states:", state_summary["state_code"].nunique())
print("Missing state abbreviations:", state_summary["state_code"].isna().sum())

# Display the first 10 states and their participation rates
print("\nState investment participation rates:")
print(state_summary.head(10).round(2))


In [ ]:

# Summarize investment participation rates across states
print("Participation rate summary:")
print(state_summary["participation_rate"].describe().round(2))

# Display the 10 states with the highest participation rates
print("\nTop 10 states:")
print(
    state_summary.nlargest(10, "participation_rate")[
        ["state_code", "participation_rate"]
    ].round(2)
)

# Display the 10 states with the lowest participation rates
print("\nBottom 10 states:")
print(
    state_summary.nsmallest(10, "participation_rate")[
        ["state_code", "participation_rate"]
    ].round(2)
)


In [ ]:

# Divide states into three groups based on participation rates
state_summary["participation_category"] = pd.qcut(
    state_summary["participation_rate"],
    q=3,
    labels=["Low", "Moderate", "High"]
)

# Verify the number of states in each category
print("Participation categories:")
print(state_summary["participation_category"].value_counts().sort_index())

# Display the actual participation ranges for each category
print("\nParticipation rate ranges:")
print(
    state_summary.groupby(
        "participation_category", observed=True
    )["participation_rate"].agg(["min", "max"]).round(2)
)


In [ ]:

# Import Plotly Express for geographic visualization
import plotly.express as px

# Create a U.S. map of investment participation
fig = px.choropleth(
    state_summary,
    locations="state_code",
    locationmode="USA-states",
    color="participation_category",
    scope="usa",
    color_discrete_map={
        "Low": "#DC2626",
        "Moderate": "#8B5CF6",
        "High": "#2563EB"
    },
    category_orders={
        "participation_category": ["High", "Moderate", "Low"]
    },
    hover_name="state_name",
    hover_data={
        "state_code": False,
        "participation_category": False,
        "participation_rate": ":.1f"
    },
    labels={
        "participation_category": "Participation Level",
        "participation_rate": "Investment Ownership (%)"
    },
    title="Non-Retirement Investment Ownership by U.S. State (2024)"
)

# Add state abbreviations (FL, TX, ...) on top of each state
fig.add_scattergeo(
    locations=state_summary["state_code"],
    locationmode="USA-states",
    text=state_summary["state_code"],
    mode="text",
    textfont=dict(color="white", size=8),
    hoverinfo="skip",
    showlegend=False
)

# Note on interpretation: groups are relative, estimates are uncertain
fig.update_layout(
    title=dict(
        subtitle=dict(
            text="Groups are relative terciles; state estimates have a "
                 "margin of error of about ±4–5 percentage points."
        )
    )
)

# Display the map
fig.show()


In [ ]:

# Save the map as a static image and an interactive HTML file
fig.write_image(
    "../visualizations/ownership_map.png", width=1200, height=700, scale=2
)
fig.write_html(
    "../visualizations/ownership_map.html", include_plotlyjs="cdn"
)
print("Saved map to ../visualizations/")